# Project 13 · Voice Support Line for Orbita Cloud (Starter notebook)

**Module 13 · Multimodal & Voice** · Part 2 (Pro) · elective

| | |
|---|---|
| ⏱️ Estimated time | 4–6 hours |
| 📋 Prerequisites | Lessons 13.02–13.05 and Project 06 (the cited knowledge assistant) |
| 🔑 Keys | none offline · `GROQ_API_KEY` (Whisper, reader, Orpheus) and `GOOGLE_API_KEY` (Gemini TTS, free tier) for the live run |
| 💸 Cost | $0 offline; live: about 10–15K Groq tokens, about 1 minute of Whisper audio, 6 Gemini TTS requests (+6 the first time) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |
| 📄 Spec | [project/README.md](README.md) |

**The job:** Orbita Cloud's support line takes voicemails all night. You build the service that answers them: transcribe the message, refuse to guess on a bad line, answer from the product documentation with verified citations, reply in the caller's language with a voice that speaks it, survive a TTS outage, and file a ticket with the transcript, the answer, its sources and per-stage timings. Then you write the report your support lead reads before switching it on.

This is the **starter**. Every `# TODO` marks work for you. The notebook runs top to bottom even before you finish: unfinished parts show ❌ in the checks instead of crashing. Stuck? Each TODO has a collapsed hint; the [reference solution](solutions/13-project-voice-support-line-solution.ipynb) is the last resort.

## Requirements (from the spec)

1. **Pipeline:** batch STT (Whisper turbo, hint terms) → ask to repeat on a bad line → translate non-English questions for retrieval → retrieve from the Project 06 corpus → answer with verified citations → speak (Gemini, Orpheus as English backup, text last) → ticket.
2. **Evidence:** 6 test calls (5 answerable, 1 noisy), a TTS-outage drill, a latency table against a budget, and an 8-check acceptance run.
3. **Report:** Markdown for the support lead, with the batch-pipeline, preview-model and privacy notes.

| TODO | Where | What |
|---|---|---|
| 1 | Step 2 | `needs_repeat(transcript)`: don't guess on a bad line |
| 2 | Step 2 | `answer_call(question, lang, call)`: retrieve, read, keep only verified citations |
| 3 | Step 2 | `spoken_reply(answer, lang)`: listenable text, callback when unknown |
| 4 | Step 2 | `speakers_for(lang)`: a voice that speaks the caller's language |
| 5 | Step 5 | `write_report(path, evidence)` |

---
## Step 0 · Setup

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/13-multimodal-and-voice/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'GOOGLE_API_KEY')
LANGSMITH_PROJECT = 'm13-project'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_GOOGLE_KEY = is_set("GOOGLE_API_KEY") or is_set("GEMINI_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m13-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast)
print("vision    :", settings.model_vision, "(Groq, preview)")
print("multimodal:", settings.model_multimodal)
print("stt / tts :", settings.model_stt, "/", settings.tts_provider)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_GOOGLE_KEY:
    print("⚠️ No GOOGLE_API_KEY: 💸 Gemini cells will print ⏭️ Skipped (free key: https://aistudio.google.com/apikey).")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models (qwen3.8 vision, Orpheus) warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Step 1 · The calls, the index and the voices

Six synthetic voicemails (`data/samples/multimodal/project13-calls.json`): four English questions, one Spanish, and one caller on a terrible line. The answers live in the Project 06 corpus, indexed here with BM25 (offline, no embeddings to download).

In [ ]:
# The pieces: the Project 06 corpus (BM25 index, offline), the test calls, and the Module 13 voice helpers.
import json
import re
import time

from mastery.rag import CitedAnswer, NOT_FOUND, extractive_answer, format_context, is_dont_know, load_corpus, split_documents, verify_citations
from mastery.rag.retrieve import BM25
from mastery.voice import (FakeSpeaker, FakeTranscriber, LatencyBudget, TtsBudget, cached_audio, flag_segments, for_voice,
                           gemini_tts, normalize_for_speech, read_wav, speak_with_fallback, split_for_tts, with_noise)

OUT_DIR = REPO_ROOT / "data" / "outputs" / "m13"           # git-ignored
AUDIO_DIR = OUT_DIR / "audio"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
SPEC = json.loads((REPO_ROOT / "data" / "samples" / "multimodal" / "project13-calls.json").read_text(encoding="utf-8"))
CALLS = {c["id"]: c for c in SPEC["calls"]}
ANSWERABLE = [cid for cid, c in CALLS.items() if "expected_source" in c]

chunks = split_documents(load_corpus(REPO_ROOT / "data" / "samples" / "rag" / "corpus"), preset="markdown",
                         chunk_size=800, chunk_overlap=100)
index = BM25(chunks)
print(f"{len(CALLS)} calls ({len(ANSWERABLE)} answerable), {len(chunks)} chunks in the Project 06 index")


def question_of(text: str) -> str | None:
    """The caller's last question ('...?', Spanish '¿...?' too), or None if they asked nothing."""
    found = re.findall(r"[^.!?¿]*\?", text)
    return found[-1].strip() if found else None


REPEAT = {"en": "Sorry, the line broke up and I couldn't hear your question. Could you say it again?",
          "es": "Disculpe, la línea se cortó y no escuché su pregunta. ¿Puede repetirla?"}
LANG_NAME = {"en": "English", "es": "Spanish"}
print(question_of("Hi, my station rebooted. At what time do updates install? Thanks."))

**You should now see** `6 calls (5 answerable), … chunks in the Project 06 index`, then `At what time do updates install?`.

> 💸 Calls Gemini on the **free tier** (no charge, daily request caps shown in AI Studio). Free-tier content may be used by Google to improve its products: the course sends synthetic samples only. Without `GOOGLE_API_KEY` it prints `⏭️ Skipped`. Orpheus needs its terms accepted once (Lesson 13.03, Step 0); until then it falls back.

In [ ]:
# 💸 The callers' audio: synthesized once with Gemini TTS (free tier, cached), the noisy call mixed at -5 dB SNR.
# Without keys: SIMULATED Whisper responses from the JSON (clearly labelled) stand in.
from mastery.voice import GeminiSpeaker, GroqTranscriber, OrpheusSpeaker

clips = {}
if HAVE_GROQ_KEY:
    for cid, call in CALLS.items():
        path = AUDIO_DIR / f"p13-{cid}.wav"
        if path.is_file() or HAVE_GOOGLE_KEY:
            data, _ = cached_audio(path, lambda c=call: gemini_tts(c["script"], voice=c["voice"], style=c["style"], sample_rate=16_000))
            clip = read_wav(data)
            clips[cid] = with_noise(clip, call["noise_snr_db"], seed=13) if "noise_snr_db" in call else clip
LIVE = HAVE_GROQ_KEY and len(clips) == len(CALLS)

if LIVE:
    stt = GroqTranscriber()                                           # whisper-large-v3-turbo
    translator = GroqTranscriber("whisper-large-v3", translate=True)  # Spanish audio → English text for retrieval
    print(f"LIVE: {len(clips)} recordings, {sum(c.seconds for c in clips.values()):.0f} s of audio")
else:
    stt = FakeTranscriber({cid: c["simulated_whisper"] for cid, c in CALLS.items()})
    translator = FakeTranscriber({cid: c["simulated_translation"] for cid, c in CALLS.items() if "simulated_translation" in c})
    print("⏭️ Skipped: needs GROQ_API_KEY (and GOOGLE_API_KEY the first time) for real audio; SIMULATED transcripts stand in.")

ORPHEUS_BUDGET = TtsBudget()
SPEAKERS = {}
if HAVE_GOOGLE_KEY:
    SPEAKERS["gemini"] = GeminiSpeaker(sample_rate=16_000)
if HAVE_GROQ_KEY:
    SPEAKERS["orpheus"] = OrpheusSpeaker(budget=ORPHEUS_BUDGET)   # needs its terms accepted once (Lesson 13.03)
if not SPEAKERS:
    SPEAKERS = {"gemini": FakeSpeaker("simulated-gemini", max_chars=2_000), "orpheus": FakeSpeaker("simulated-orpheus")}
print("speakers:", {k: v.name for k, v in SPEAKERS.items()})


def read_answer(question: str, docs: list, lang: str, call: dict) -> CitedAnswer:
    """The reader: Groq's fast model with structured output; offline, the extractive reader from Module 06."""
    if HAVE_GROQ_KEY:
        from mastery.models import get_model
        prompt = (f"You answer phone callers for Orbita Cloud. Answer in {LANG_NAME[lang]}, in at most two short sentences, "
                  "using ONLY the sources below. Every citation quotes the source word for word (the sources are in English). "
                  f"If the sources do not answer the question, answer exactly: {NOT_FOUND}\n\n{format_context(docs)}\n\nQuestion: {question}")
        return get_model("fast", temperature=0).with_structured_output(CitedAnswer).invoke(prompt)
    answer = extractive_answer(question, docs)
    if lang != "en" and not is_dont_know(answer) and call.get("simulated_answer"):
        answer = CitedAnswer(answer=call["simulated_answer"], citations=answer.citations)   # SIMULATED wording offline
    return answer

**You should now see** `LIVE: 6 recordings, … s of audio` with keys, or `⏭️ Skipped: …; SIMULATED transcripts stand in`, then the speakers in use.

---
## Step 2 · Your four functions

**TODO 1** is in the cell below.

> 💡 **Hint:** Return a reason string when `flag_segments(transcript)` (Lesson 13.02) flags any segment, or when `question_of(transcript.text)` finds no question; otherwise `None`. A noisy line where Whisper wrote "Thank you." with a high `no_speech_prob` must not reach the agent.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def needs_repeat(transcript) -> str | None:
    """Return why we should ask the caller to repeat (a short reason), or None if we can answer."""
    flags = flag_segments(transcript)              # no speech + low confidence, loops, low confidence
    if flags:
        return flags[0][1]
    if question_of(transcript.text) is None:
        return "no question heard"
    return None
```

</details>

**TODO 2** is in the cell below.

> 💡 **Hint:** `index.search(question, k=5)` returns chunks; `read_answer(question, docs, lang, call)` returns a `CitedAnswer`; `verify_citations(answer, docs)` (Module 06) returns `{"valid", "fabricated", "ok"}`. Return the answer rebuilt with only the valid citations.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def answer_call(question: str, lang: str, call: dict) -> tuple[CitedAnswer, list, dict]:
    """Retrieve, read, verify: (answer with ONLY verified citations, retrieved docs, verify_citations result)."""
    docs = index.search(question, k=5)
    answer = read_answer(question, docs, lang, call)
    check = verify_citations(answer, docs)
    answer = CitedAnswer(answer=answer.answer, citations=check["valid"])     # never file an unverified quote
    return answer, docs, check
```

</details>

**TODO 3** is in the cell below.

> 💡 **Hint:** Use `is_dont_know(answer)` to return a callback promise in the caller's language; otherwise `for_voice(..., max_sentences=2)` (13.04) and `normalize_for_speech` (13.03), which drop `**` and file citations.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
CALLBACK = {"en": "I couldn't find that in our documentation. I've opened a ticket and a person will call you back today.",
            "es": "No encontré eso en nuestra documentación. Abrí un ticket y una persona le devolverá la llamada hoy."}


def spoken_reply(answer: CitedAnswer, lang: str) -> str:
    """What the caller hears: short, no Markdown, no file names; a callback promise when we don't know."""
    if is_dont_know(answer):
        return CALLBACK[lang]
    return normalize_for_speech(for_voice(answer.answer, max_sentences=2))
```

</details>

**TODO 4** is in the cell below.

> 💡 **Hint:** Return Gemini then Orpheus for `"en"` (whichever exist in `SPEAKERS`), and only Gemini for any other language: Orpheus's voices are English-only (Lesson 13.03).

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def speakers_for(lang: str) -> list:
    """The speaker chain for this language (speak_with_fallback ends in text anyway)."""
    if lang == "en":
        return [SPEAKERS[name] for name in ("gemini", "orpheus") if name in SPEAKERS]
    return [SPEAKERS["gemini"]] if "gemini" in SPEAKERS else []     # Orpheus: English voices only
```

</details>

In [ ]:
def needs_repeat(transcript) -> str | None:
    """Return why we should ask the caller to repeat (a short reason), or None if we can answer."""
    # TODO 1: don't guess on a bad line. Flag Whisper's own warning signs and "no question heard".
    return None

In [ ]:
def answer_call(question: str, lang: str, call: dict) -> tuple[CitedAnswer, list, dict]:
    """Retrieve, read, verify: (answer with ONLY verified citations, retrieved docs, verify_citations result)."""
    # TODO 2: retrieve 5 chunks, read them with read_answer, and keep only the citations that verify.
    return CitedAnswer(answer=NOT_FOUND, citations=[]), [], {"ok": False, "valid": [], "fabricated": []}

In [ ]:
def spoken_reply(answer: CitedAnswer, lang: str) -> str:
    """What the caller hears: short, no Markdown, no file names; a callback promise when we don't know."""
    # TODO 3: make the answer listenable (and handle "don't know").
    return answer.answer

In [ ]:
def speakers_for(lang: str) -> list:
    """The speaker chain for this language (speak_with_fallback ends in text anyway)."""
    # TODO 4: Orpheus speaks English only. Choose a chain per language.
    return [SPEAKERS["orpheus"]] if "orpheus" in SPEAKERS else []

---
## Step 3 · Run the line

> 💸 With keys: 6 Whisper requests (+1 translation), 5 reader calls on `gpt-oss-20b`, 6 Gemini TTS requests.

In [ ]:
# 💸 (LIVE when HAVE_GROQ_KEY) Every call through the pipeline, then one ticket in full.
# One call, batch pipeline: STT → (ask to repeat | translate → retrieve → read → verify) → speak → ticket.
def handle_call(cid: str, speakers: list | None = None) -> tuple[dict, object]:
    call, t = CALLS[cid], {}
    start = time.perf_counter()
    heard = stt.transcribe(clips.get(cid), key=cid, prompt=", ".join(SPEC["hint_terms"]))
    t["stt"] = (time.perf_counter() - start) * 1000
    lang = "es" if (heard.language or "").casefold() in ("es", "spanish") else "en"
    ticket = {"call": cid, "lang": lang, "transcript": heard.text.strip(), "simulated": heard.simulated}
    start = time.perf_counter()
    reason = needs_repeat(heard)
    if reason:
        reply = REPEAT[lang]
        ticket.update(route="ask_to_repeat", reason=reason)
    else:
        question = question_of(heard.text) or heard.text.strip()
        if lang != "en":                                   # retrieve in the corpus language
            english = translator.transcribe(clips.get(cid), key=cid)
            question = question_of(english.text) or english.text.strip()
        answer, docs, check = answer_call(question, lang, call)
        reply = spoken_reply(answer, lang)
        ticket.update(route="callback" if is_dont_know(answer) else "answered", question=question, answer=answer.answer,
                      citations=[{"source": c.source, "page": c.page, "quote": c.quote} for c in answer.citations],
                      fabricated=len(check.get("fabricated", [])))
    t["answer"] = (time.perf_counter() - start) * 1000
    speech = speak_with_fallback(reply, speakers if speakers is not None else speakers_for(lang))
    t["tts"] = speech.total_s * 1000
    t["first_audio"] = t["stt"] + t["answer"] + speech.first_audio_s * 1000
    ticket.update(spoken=reply, speech=speech.provider, speech_errors=speech.errors,
                  timings_ms={k: round(v) for k, v in t.items()})
    if speech.wav and not speech.simulated:
        (AUDIO_DIR / f"p13-answer-{cid}.wav").write_bytes(speech.wav)
    return ticket, speech


tickets = {}
for cid in CALLS:
    tickets[cid], _ = handle_call(cid)
table([{"call": k, "lang": v["lang"], "route": v["route"], "speech": v["speech"], **v["timings_ms"],
        "spoken": v["spoken"][:60]} for k, v in tickets.items()],
      ["call", "lang", "route", "speech", "stt", "answer", "tts", "first_audio", "spoken"])
print(json.dumps(tickets["call-02"], ensure_ascii=False, indent=1)[:700])

**You should now see** (solution) a table with five `answered` calls and `call-06-noisy` as `ask_to_repeat`, the Spanish call spoken by Gemini (or the simulated Gemini) in Spanish, and the full ticket for `call-02` with its citation from `support/faq-app-login.md`. Starter: `callback` everywhere and Orpheus for the Spanish call.

---
## Step 4 · Failure drill and latency

In [ ]:
# 🧪 A TTS outage mid-call: every speaker fails, the caller must still get the answer (as text / SMS).
outage, outage_speech = handle_call("call-02", speakers=[FakeSpeaker("tts-outage", fail="503 Service Unavailable")])
print(outage["route"], "| speech:", outage["speech"], "| errors:", outage["speech_errors"])
print("text answer:", outage["spoken"])

**You should now see** `answered | speech: text-only | errors: ['tts-outage: RuntimeError: 503 Service Unavailable']` and the text answer (solution).

In [ ]:
# 🧪 Per-stage latency against a budget for this batch line (seconds matter on a phone; see Lesson 13.05).
BUDGET = LatencyBudget({"stt": 1500, "answer": 2500, "tts": 3000})
answered = [tickets[c] for c in ANSWERABLE]
mean = {stage: sum(tk["timings_ms"][stage] for tk in answered) / len(answered) for stage in ("stt", "answer", "tts")}
latency_rows = BUDGET.report(mean)
table(latency_rows, ["stage", "target ms", "measured ms", "ok"])
print("mean first audio:", round(sum(tk["timings_ms"]["first_audio"] for tk in answered) / len(answered)), "ms",
      "(SIMULATED stages)" if not LIVE else "")

**You should now see** the budget table with `stt`, `answer`, `tts` and `TOTAL`, and the mean first audio. Offline the numbers are tiny and marked `(SIMULATED stages)`; live, TTS usually dominates.

---
## Step 5 · The report

**TODO 5** is in the cell below.

> 💡 **Hint:** Write a Markdown string with the five `##` sections, a calls table from `evidence["tickets"]`, the latency rows from `evidence["latency"]`, and say in words that it's a **batch** pipeline, that the TTS models are **preview**, and that the calls are **synthetic** (privacy). Save it to `path` and return it.

<details>
<summary>Still stuck? Show one way to write it</summary>

```python
def write_report(path, evidence: dict) -> str:
    """Write the Markdown report for the support lead and return its text."""
    rows = "\n".join(f"| {t['call']} | {t['lang']} | {t['route']} | {t['speech']} | {t['timings_ms']['first_audio']} | "
                     f"{', '.join(c['source'] for c in t.get('citations', [])) or '-'} |" for t in evidence["tickets"].values())
    lat = "\n".join(f"| {r['stage']} | {r['target ms']} | {r['measured ms']} | {'✅' if r['ok'] else '❌'} |" for r in evidence["latency"])
    text = f"""# Project 13 voice support line report

## Pipeline

A **batch** pipeline: the whole voicemail is recorded, transcribed with Groq Whisper, answered from the Project 06
documentation with verified citations, spoken with Gemini TTS (Orpheus as English backup) and filed as a ticket.
Mode: {"LIVE" if evidence["live"] else "SIMULATED (offline)"}.

## Calls

| Call | Lang | Route | Speech | First audio ms | Citations |
|---|---|---|---|---|---|
{rows}

TTS outage drill: route `{evidence["outage"]["route"]}`, speech `{evidence["outage"]["speech"]}` (text answer still returned).

## Latency

| Stage | Target ms | Measured ms | OK |
|---|---|---|---|
{lat}

## Production notes

- Orpheus and Gemini TTS are **preview** models: pin versions, keep the text fallback, watch daily quotas.
- Real phone lines need streaming STT/TTS, endpointing and barge-in (Lesson 13.05); this batch line suits voicemail.
- Low-confidence audio is routed to "ask to repeat"; unknown questions become a callback ticket, never a guess.

## Privacy

All calls are **synthetic**. Real calls are personal data: announce the assistant and recording, keep transcripts
(not audio) for the retention period, and don't send real customer audio to free tiers that may train on it.
"""
    path.write_text(text, encoding="utf-8")
    return text
```

</details>

In [ ]:
def write_report(path, evidence: dict) -> str:
    """Write the Markdown report for the support lead and return its text."""
    # TODO 5: sections Pipeline, Calls, Latency, Production notes, Privacy (see the spec).
    text = "# TODO\n"
    path.write_text(text, encoding="utf-8")
    return text

In [ ]:
evidence = {"tickets": tickets, "outage": outage, "latency": latency_rows, "live": LIVE}
report = write_report(OUT_DIR / "project13-report.md", evidence)
(OUT_DIR / "project13-tickets.json").write_text(json.dumps(tickets, ensure_ascii=False, indent=1), encoding="utf-8")
print(report[:900])

**You should now see** the report starting `# Project 13 voice support line report` (solution). Saved with the tickets to `data/outputs/m13/`.

---
## Acceptance checks

In [ ]:
SPANISH = {"el", "la", "de", "que", "y", "en", "los", "las", "para", "con", "una", "un", "tiene", "días", "su", "desde", "si", "no", "es"}


def has_fact(tk):
    return any(x.casefold() in tk.get("answer", "").casefold() for x in CALLS[tk["call"]]["answer_contains"])


def chunks_ok(text):
    parts = split_for_tts(text)
    return bool(parts) and all(len(p) <= 200 for p in parts) and " ".join(parts).split() == normalize_for_speech(text).split()


def accept(name, ok):
    results[name] = bool(ok)
    print(("✅" if ok else "❌"), name)


results = {}
answered = [tickets[c] for c in ANSWERABLE]
es = next(t for t in answered if t["lang"] == "es")
accept("5 calls answered end to end with the right fact",
       all(t["route"] == "answered" and has_fact(t) for t in answered))
accept("TTS chunks respect 200 characters and never cut words", all(chunks_ok(t["spoken"]) for t in tickets.values()))
accept("per-stage latency table against the budget",
       all({"stt", "answer", "tts", "first_audio"} <= set(t["timings_ms"]) for t in tickets.values())
       and [r["stage"] for r in latency_rows] == ["stt", "answer", "tts", "TOTAL"])
accept("TTS outage still returns the text answer",
       outage["speech"] == "text-only" and outage["route"] == "answered" and has_fact(outage))
accept("noisy call is routed to ask-to-repeat, not answered",
       tickets["call-06-noisy"]["route"] == "ask_to_repeat" and "answer" not in tickets["call-06-noisy"])
accept("Spanish question answered in Spanish, not by an English-only voice",
       len(SPANISH & set(re.findall(r"\w+", es["spoken"].casefold()))) >= 3 and "orpheus" not in es["speech"])
accept("verified citations from the right document, none spoken",
       all(t["citations"] and t["fabricated"] == 0 and CALLS[t["call"]]["expected_source"] in {c["source"] for c in t["citations"]}
           and not re.search(r"\.(md|pdf)\b|\*\*", t["spoken"]) for t in answered))
accept("report: sections plus batch, preview and privacy notes",
       all(s in report for s in ("## Pipeline", "## Calls", "## Latency", "## Production notes", "## Privacy"))
       and all(w in report.casefold() for w in ("batch", "preview", "synthetic")))
print(f"\n{sum(results.values())}/{len(results)} acceptance checks passed")
(OUT_DIR / "project13-acceptance.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

**You should now see** `8/8 acceptance checks passed` (solution). Starter: `2/8` (the chunking and the latency table come from the course library; everything else needs a TODO). Results: `data/outputs/m13/project13-acceptance.json`.

## Findings (write yours)

- Which stage dominated first audio live, and what would you change first (Lesson 13.05)?
- Listen to the Spanish answer: is the voice right for your callers? What would you test with real customers?
- Which call would you not trust this line with yet, and why?

## ✅ Before you commit

- [ ] Restart & Run All passes without a key.
- [ ] All TODOs done; 8/8 acceptance checks; report and findings written.
- [ ] Outputs cleared (`jupyter nbconvert --clear-output --inplace <notebook>`); `data/outputs/` not committed.

---
## ➡️ What's next

🎉 **Module 13 done:** documents, images and voice now feed your agents.

**Next:** see [COURSE_OUTLINE.md](../../../COURSE_OUTLINE.md) for the following module. Back to the [Module 13 README](../../../modules/13-multimodal-and-voice/README.md).